# Enrich Shifts

Correct overnight exits, calculate durations, and classify long, holiday, afternoon, and night shifts.

**Requires:** employee `*.pairs.csv` files.  
**Produces:** employee `*.enriched.csv` files and an enrichment report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("enrich_shifts")
display(nb.pipeline_overview(ctx, "enrich_shifts"))


## Controls


In [ ]:
VERBOSE = True
MIN_HOURS = float(step_cfg.get("min_hours", 6.0))
INCLUDE_HOLIDAYS = bool(step_cfg.get("include_holidays", True))

{
    "min_hours": MIN_HOURS,
    "include_holidays": INCLUDE_HOLIDAYS,
}


## Input Preview


In [ ]:
display(nb.file_table(paths.shifts_dir, "*.pairs.csv"))
pair_files = sorted(paths.shifts_dir.glob("*.pairs.csv"))
if pair_files:
    display(nb.preview_csv(pair_files[0]))


## Build Options


In [ ]:
from core.shifts.enrichment.options import TurniEnrichmentOptions

options = TurniEnrichmentOptions(
    input_dir=str(paths.shifts_dir),
    output_dir=str(paths.enrichment_dir),
    min_hours=MIN_HOURS,
    include_holidays=INCLUDE_HOLIDAYS,
    report_json=str(paths.enrichment_report),
    verbose=VERBOSE,
)
options


## Run Enrichment


In [ ]:
from core.drive.logging_utils import setup_logging
from core.shifts.enrichment.service import run_from_options

setup_logging(VERBOSE)
enrichment_report = run_from_options(options)
display(nb.report_summary(enrichment_report))


## Inspect Employee Files


In [ ]:
display(nb.artifact_table({"enrichment report": paths.enrichment_report}))
display(nb.file_table(paths.enrichment_dir, "*.enriched.csv"))
enriched_files = sorted(paths.enrichment_dir.glob("*.enriched.csv"))
if enriched_files:
    display(nb.preview_csv(enriched_files[0]))
